In [ ]:
# ===== 環境セットアップ =====
import os, subprocess, glob

def _is_kaggle():
    return os.path.exists("/kaggle/input")

def _is_runpod():
    return os.environ.get("RUNPOD_POD_ID") is not None

ENV = "kaggle" if _is_kaggle() else "runpod" if _is_runpod() else "local"
print(f"[env] {ENV}")

if ENV == "runpod":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import json as _json
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": os.environ["KAGGLE_USERNAME"],
                    "key": os.environ["KAGGLE_KEY"]}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
# ===== パス定義 =====

def _find_path(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return candidates[0]

def _find_adapter_path():
    """adapter_config.json で実際のマウントパスを検索"""
    # 自分のアダプタを優先検索
    for pattern in [
        "/kaggle/input/models/shotokishida/nemotron-adapter/*/*/*",
        "/kaggle/input/*nemotron-adapter*",
    ]:
        hits = glob.glob(pattern)
        if hits:
            return hits[0]
    # adapter_config.json で汎用検索（fallback）
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("adapter_config.json が見つかりません。model_sources を確認してください。")

if ENV == "kaggle":
    ADAPTER_PATH         = _find_adapter_path()
    NEMOTRON_MODEL_DIR   = _find_path(
                               "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
                               "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1")
    METRIC_UTIL_DIR      = "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script"
    COMPETITION_DATA_DIR = _find_path(
                               "/kaggle/input/nvidia-nemotron-model-reasoning-challenge",
                               "/kaggle/input/nvidia-nemotron-3-reasoning-challenge")
    OUTPUT_DIR           = "/kaggle/working"
elif ENV == "runpod":
    ADAPTER_PATH         = "/workspace/models/nemotron-adapter"
    NEMOTRON_MODEL_DIR   = "/workspace/models/nemotron-3-nano-30b-a3b-bf16"
    METRIC_UTIL_DIR      = "/workspace/data/nvidia-metric-utility-script"
    COMPETITION_DATA_DIR = "/workspace/data/nvidia-nemotron-competition"
    OUTPUT_DIR           = "/workspace/output"
else:
    ADAPTER_PATH         = "models/nemotron-adapter"
    NEMOTRON_MODEL_DIR   = None
    METRIC_UTIL_DIR      = "data/nvidia-metric-utility-script"
    COMPETITION_DATA_DIR = "data/nvidia-nemotron-competition"
    OUTPUT_DIR           = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"ADAPTER_PATH={ADAPTER_PATH}  exists={os.path.exists(ADAPTER_PATH)}")
print(f"NEMOTRON_MODEL_DIR={NEMOTRON_MODEL_DIR}  exists={os.path.exists(str(NEMOTRON_MODEL_DIR))}")
print(f"COMPETITION_DATA_DIR={COMPETITION_DATA_DIR}  exists={os.path.exists(COMPETITION_DATA_DIR)}")

In [ ]:
TEST_GENERATION = True  # False にすると vLLM 推論をスキップして submission.zip のみ作成

In [ ]:
# ===== アダプタ変換 =====
# PEFT が保存するキー名を vLLM が期待する形式に変換する。
#   ① キーリネーム: base_model.model.model.* → base_model.model.backbone.*
#   ② MoE expert unfuse: experts.w1/w2 → experts.N.up_proj/down_proj
#      （PEFT が MoE の shared weight として保存する場合に個別 expert へ展開）
import re
import shutil
import json
import torch
from safetensors import safe_open
from safetensors.torch import save_file

def convert_adapter(src_dir: str, dst_dir: str):
    os.makedirs(dst_dir, exist_ok=True)

    # adapter_config.json をコピー（target_modules はそのまま使用）
    shutil.copy2(os.path.join(src_dir, "adapter_config.json"),
                 os.path.join(dst_dir, "adapter_config.json"))
    with open(os.path.join(dst_dir, "adapter_config.json")) as f:
        cfg = json.load(f)
    print(f"[adapter] target_modules: {cfg.get('target_modules')}")
    print(f"[adapter] lora_rank: {cfg.get('r')}")

    # adapter_model.safetensors を変換
    src_st = os.path.join(src_dir, "adapter_model.safetensors")
    raw = {}
    with safe_open(src_st, framework="pt", device="cpu") as f:
        for key in f.keys():
            raw[key] = f.get_tensor(key)
    print(f"[adapter] loaded {len(raw)} tensors from {src_st}")

    # base_names 収集
    base_names = set()
    for key in raw:
        base = re.sub(r"\.lora_[AB]\.weight$", "", key)
        base_names.add(base)

    tensors = {}
    for base in sorted(base_names):
        lora_A = raw[f"{base}.lora_A.weight"]
        lora_B = raw[f"{base}.lora_B.weight"]

        # ① キーリネーム: model.model → model.backbone
        renamed = base.replace("base_model.model.model.", "base_model.model.backbone.")

        # ② MoE expert unfuse: experts.w1/w2 → experts.N.up_proj/down_proj
        if ".experts.w1" in base or ".experts.w2" in base:
            if lora_A.shape[0] == 1:
                lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
            elif lora_B.shape[0] == 1:
                lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()

            num_experts = lora_A.shape[0]
            proj_name = "up_proj" if ".w1" in base else "down_proj"
            for i in range(num_experts):
                exp_renamed = re.sub(r"\.experts\.w[12]",
                                     f".experts.{i}.{proj_name}", renamed)
                tensors[f"{exp_renamed}.lora_A.weight"] = lora_A[i].contiguous()
                tensors[f"{exp_renamed}.lora_B.weight"] = lora_B[i].contiguous()
            continue

        # ③ 空テンソルのスキップ（例: 空の expert）
        if lora_A.numel() == 0:
            continue

        tensors[f"{renamed}.lora_A.weight"] = lora_A
        tensors[f"{renamed}.lora_B.weight"] = lora_B

    dst_st = os.path.join(dst_dir, "adapter_model.safetensors")
    save_file(tensors, dst_st)
    print(f"[adapter] saved {len(tensors)} tensors → {dst_st}")

convert_adapter(ADAPTER_PATH, OUTPUT_DIR)
print("[adapter] conversion done")

In [ ]:
import glob
import math
import multiprocessing
import os
import re
import time
from pathlib import Path

import kagglehub
import pandas as pd
from tqdm import tqdm

# Configuration
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
DATA_PATH = (
    ""  # Path(kagglehub.dataset_download('metric/nvidia-nemotron-rerun-data-129716'))
)

In [ ]:
# ===== Triton / torch セットアップ（Blackwell GPU 対応） =====
import subprocess, sys, math, multiprocessing, re, time
from pathlib import Path

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    f"tar -cf - -C {METRIC_UTIL_DIR} . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]
if TEST_GENERATION:
    for cmd in commands:
        print(f"Running: {cmd}")
        subprocess.run(cmd, shell=True, check=True)
sys.path.insert(0, "/tmp")

In [ ]:
# ===== ヘルパー関数 =====
import kagglehub
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"MODEL_PATH={MODEL_PATH}")

def cache_model(path, exts=(".bin", ".pt", ".safetensors"), num_workers=None, chunk_mb=256):
    def warmup_file(fpath):
        total = 0
        try:
            with open(fpath, "rb") as f:
                while chunk := f.read(chunk_mb * 1024 * 1024):
                    total += len(chunk)
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
        return fpath, total
    path = Path(path)
    files = sorted([p for p in path.rglob("*") if p.is_file() and str(p).endswith(exts)]) if path.is_dir() else ([path] if path.exists() else [])
    if not files:
        print(f"No model files found: {path}")
        return 0
    num_workers = num_workers or min(multiprocessing.cpu_count(), 8)
    print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
    t0 = time.time()
    total_bytes = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for i, fut in enumerate(as_completed({pool.submit(warmup_file, f): f for f in files}), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f"[{i}/{len(files)}] cached {fpath.name}")
    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    print(f"[cache_model] total ≈ {gb:.2f} GB in {elapsed:.2f}s ({gb/elapsed:.2f} GB/s)")
    return total_bytes

def extract_final_answer(text):
    if text is None:
        return "NOT_FOUND"
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()
    for pattern in [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"

def verify(stored_answer, predicted):
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    try:
        return math.isclose(float(stored_answer), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()

In [ ]:
# ===== モデルキャッシュ + vLLM 初期化 =====
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

# 推論パラメータ（competition 評価と同じ設定）
max_model_len           = 8192
max_lora_rank           = 32    # competition の上限
max_tokens              = 7680
top_p                   = 1.0
temperature             = 0.0
max_num_seqs            = 64
gpu_memory_utilization  = 0.85

if TEST_GENERATION:
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )
    sampling_params = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_tokens)
    print("[vllm] initialized")

In [ ]:
# ===== プロンプト準備 =====
# lora_path: 変換済みアダプタのパス（OUTPUT_DIR に adapter_config.json が存在する）
lora_path = OUTPUT_DIR
print(f"lora_path={lora_path}  adapter_config exists={os.path.exists(os.path.join(lora_path, 'adapter_config.json'))}")

if TEST_GENERATION:
    tokenizer = llm.get_tokenizer()

    # ローカルテスト用サブセット（train.csv で正解が確認できる問題を使用）
    df = pd.read_csv(f"{COMPETITION_DATA_DIR}/train.csv")
    problem_set = {
        "836b85e8", "b20b39bf", "af358750", "3f9bd1e7", "0528d502", "9992bbd0", "812131f1", "84e3f9f7",
        "49b244e3", "7b8e4432", "3019f44e", "4f8f23d6", "1fcbbb93", "987a223b", "84d10c70", "0dad87bf",
        "1c7a0091", "ed61a9d6", "02b8d816", "d6c03e21", "2f6531cb", "a4ee9fa6", "02a04b59", "81b6d789",
        "c7844441", "0fcf912a", "07b440f0", "25ee72c3", "9dfe5ac9", "deed3497", "258b796b", "0da1841f",
        "91488dc9", "7e2e8a95", "35a89469", "a04ecffd", "27cec7a9", "d6a2e332", "04322d27", "8bc6a26c",
        "662fd21c", "e9e6b620", "8ae8e12a", "dc178d1c", "9c91b226", "c763054a", "66a0856f", "ef1b13ac",
        "d0dd2df7", "74a50b2c", "f9f20a7a", "ef3c7703", "85bc954c", "8de7d8bc", "22bb13b8", "87eb7ce0",
        "3b4ebafd", "ad6ff612", "5a6ed2bf", "0adca57b", "d79d0cfd", "e6b04620", "f47276a4", "1e2de753",
        "082c1a06", "3dcaf042", "87342969", "8e1cff16", "d566ff0e", "598af975", "51a22965", "d3d82844",
    }
    df = df[df.id.isin(problem_set)].copy()

    prompts = []
    for problem_text in df["prompt"]:
        user_content = problem_text + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
        try:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False, add_generation_prompt=True, enable_thinking=True,
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False, add_generation_prompt=True,
            )
        prompts.append(prompt)
    print(f"[prompts] {len(prompts)} problems prepared")

In [ ]:
# ===== submission.zip 作成 =====
# アダプタ2ファイルを submission.zip にまとめる
import zipfile as _zf

required = ["adapter_config.json", "adapter_model.safetensors"]
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")

with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for fname in required:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"CRITICAL: {fname} が見つかりません")
        zf.write(fpath, fname)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"submission.zip: {size_mb:.1f} MB")
print(f"contents: {required}")